In [ ]:
!pip install azure-storage-blob pandas numpy scikit-learn tensorflow

import os
import json
import pandas as pd
import numpy as np
from io import StringIO
from azure.storage.blob import BlobServiceClient
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tqdm.notebook import tqdm
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from typing import Dict, Any, List
AZURE_CONNECTION_STRING = ""


In [ ]:
import pandas as pd
import numpy as np
import json
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import io
from typing import Dict, Any, List
from concurrent.futures import ThreadPoolExecutor

try:
    from azure.storage.blob import BlobServiceClient, ContainerClient
    from io import BytesIO
except ImportError:
    class BlobServiceClient: pass
    class ContainerClient: pass
    class BytesIO: pass
    print("Warning: azure-storage-blob is not installed. Azure download logic will not execute until installed and credentials are set.")


# --- Project Configuration ---

CONNECTION_STRING = AZURE_CONNECTION_STRING
CONTAINER_NAME = "london"
SEGMENTATION_FOLDER = "SemanticSegmentation/json/" # Ensure trailing slash
DETECTION_ROOT = "YoloV10/"
CONFIDENCE_LEVELS = [0.7]

# Placeholder for the complete set of Image IDs (19995 total)
NUM_IMAGES = 19995
IMAGE_IDS = [f"{i}_{angle}.jpg" for i in range(NUM_IMAGES // 4 + 1) for angle in [0, 90, 180, 270]][:NUM_IMAGES]

SEG_FULL_CLASSES = [
    'Bicycle', 'Bicyclist', 'Bike Lane', 'Bike Rack', 'Billboard', 'Bird',
    'Boat', 'Boat Mount', 'Bridge', 'Building', 'Bus', 'Banner', 'Barrier',
    'Bench', 'Car', 'Car Mount', 'Caravan', 'Catch Basin', 'CCTV Camera',
    'Curb', 'Curb Cut', 'Crosswalk—Plain', 'Ego Vehicle', 'Fence',
    'Fire Hydrant', 'Guard Rail', 'Junction Box', 'Lane', 'Lane Marking—Crosswalk',
    'Lane Marking—General', 'Mailbox', 'Manhole', 'Mountain', 'Motorcycle',
    'Motorcyclist', 'On Rails', 'Other Rider', 'Other Vehicle', 'Parking',
    'Pedestrian Area', 'Phone Booth', 'Pole', 'Pothole', 'Rail Track',
    'Road', 'Sand', 'Service Lane', 'Sidewalk', 'Sky', 'Snow', 'Street Light',
    'Terrain', 'Traffic Light', 'Traffic Sign (Back)', 'Traffic Sign (Front)',
    'Traffic Sign Frame', 'Trailer', 'Trash Can', 'Truck', 'Tunnel', 'Unlabeled',
    'Utility Pole', 'Vegetation', 'Water', 'Wheeled Slow'
]
OBJ_FULL_CLASSES = [
    'Person', 'Bicycle', 'Car', 'Motorcycle', 'Airplane', 'Bus', 'Train', 'Truck', 'Boat', 'Traffic Light',
    'Fire Hydrant', 'Stop Sign', 'Parking Meter', 'Bench', 'Bird', 'Cat', 'Dog', 'Horse', 'Sheep', 'Cow',
    'Elephant', 'Bear', 'Zebra', 'Giraffe', 'Backpack', 'Umbrella', 'Handbag', 'Tie', 'Suitcase', 'Frisbee',
    'Skis', 'Snowboard', 'Sports Ball', 'Kite', 'Baseball Bat', 'Baseball Glove', 'Skateboard', 'Surfboard',
    'Tennis Racket', 'Bottle', 'Wine Glass', 'Cup', 'Fork', 'Knife', 'Spoon', 'Bowl', 'Banana', 'Apple',
    'Sandwich', 'Orange', 'Broccoli', 'Carrot', 'Hot Dog', 'Pizza', 'Donut', 'Cake', 'Chair', 'Couch',
    'Potted Plant', 'Bed', 'Dining Table', 'Toilet', 'Tv', 'Laptop', 'Mouse', 'Remote', 'Keyboard',
    'Cell Phone', 'Microwave', 'Oven', 'Toaster', 'Sink', 'Refrigerator', 'Book', 'Clock', 'Vase',
    'Scissors', 'Teddy Bear', 'Hair Drier', 'Toothbrush'
]

# --- Step 2 Required Class Selection ---
SELECTED_SEG_CLASSES = SEG_FULL_CLASSES
SELECTED_OBJ_CLASSES = OBJ_FULL_CLASSES

# --- Helper Functions for Parsing ---

def parse_segmentation(json_content: str) -> Dict[str, float]:
    """Parses segmentation JSON content and returns a dictionary of class names to pixel percentages."""
    try:
        data = json.loads(json_content)
        features = {}
        # Assuming a fixed image size like 640x640, or use the provided dimensions
        total_pixels = data.get('height', 640) * data.get('width', 640)

        for entry in data.get('label_histogram', []):
            # Store pixel percentage relative to the image size
            features[entry['label_name']] = entry['pixels'] / total_pixels
        return features
    except Exception as e:
        print(f"Error parsing segmentation JSON: {e}")
        return {}

def parse_yolo(txt_content: str) -> Dict[str, int]:
    """Parses YOLO TXT content and returns a dictionary of object class counts."""
    try:
        lines = txt_content.strip().split('\n')
        class_counts = {}
        for line in lines:
            parts = line.split()
            if parts:
                # YOLO format is typically: class_id confidence x_center y_center width height
                class_id = int(parts[0])
                # We map the ID to the class name using the full list
                if class_id < len(OBJ_FULL_CLASSES):
                    class_name = OBJ_FULL_CLASSES[class_id]
                    class_counts[class_name] = class_counts.get(class_name, 0) + 1
        return class_counts
    except Exception as e:
        print(f"Error parsing YOLO TXT: {e}")
        return {}

MAX_WORKERS = 32

def download_and_process_blob(blob_item, container_client, is_segmentation: bool):
    """Downloads, reads, and parses a single blob item. Returns (image_id, features) or None on error."""
    try:
        blob_name = blob_item['name'] # Use the name from the blob dictionary/item

        blob_client = container_client.get_blob_client(blob_name)
        download_stream = blob_client.download_blob()
        content = download_stream.readall().decode('utf-8')

        if is_segmentation:
            if not blob_name.endswith('.json'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_segmentation(content)
            prefixed_features = {f"seg_{k}": v for k, v in features.items()}
        else:
            if not blob_name.endswith('.txt'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_yolo(content)
            prefixed_features = {f"obj_{k}": v for k, v in features.items()}

        if image_id not in IMAGE_IDS:
            return None

        return (image_id, prefixed_features)

    except Exception as e:
        return None

# --- Step 1: Download and Combine Logic ---

def download_and_combine_data() -> Dict[str, pd.DataFrame]:
    """
    Connects to Azure Blob Storage, downloads segmentation and object detection
    data for all confidence levels using parallel processing, and combines them.
    """
    print("--- Step 1: Connecting to Azure and Downloading All Blobs (Parallelized) ---")

    if CONNECTION_STRING == "YOUR_AZURE_BLOB_STORAGE_CONNECTION_STRING_HERE":
         raise ValueError("Please replace 'YOUR_AZURE_BLOB_STORAGE_CONNECTION_STRING_HERE' with your actual Azure connection string.")

    try:
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)
    except Exception as e:
        raise ConnectionError(f"Failed to connect to Azure: {e}")

    # Initialize a master map to store all features for all images
    feature_map = {image_id: {} for image_id in IMAGE_IDS}

    # 1. Process Segmentation Data (Consistent across all confidence levels)
    print("\n[1/2] Processing Semantic Segmentation Data (JSON files)...")

    # Efficiently list all relevant segmentation blobs
    seg_blob_items = list(container_client.list_blobs(name_starts_with=SEGMENTATION_FOLDER, results_per_page=1000).by_page())
    seg_blob_items = [blob for page in seg_blob_items for blob in page] # Flatten list of pages

    seg_tasks = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for blob_item in seg_blob_items:
            seg_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, True))

        for future in tqdm(seg_tasks, desc="Downloading Segmentation (Parallel)"):
            result = future.result()
            if result:
                image_id, features = result
                feature_map[image_id].update(features)


    # 2. Process Object Detection Data for all 3 confidence levels
    all_dataframes = {}

    print("\n[2/2] Processing Object Detection (YOLO) Data (TXT files) for 3 Confidence Levels...")

    for conf in CONFIDENCE_LEVELS:
        conf_str = f"{conf:.1f}"
        detection_folder = f"{DETECTION_ROOT}conf_{conf_str}/labels/"

        # List all relevant detection blobs
        obj_blob_items = list(container_client.list_blobs(name_starts_with=detection_folder, results_per_page=1000).by_page())
        obj_blob_items = [blob for page in obj_blob_items for blob in page]

        current_conf_features = {img_id: feature_map[img_id].copy() for img_id in IMAGE_IDS}

        obj_tasks = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for blob_item in obj_blob_items:
                obj_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, False))

            for future in tqdm(obj_tasks, desc=f"Downloading YOLO {conf_str} (Parallel)"):
                result = future.result()
                if result:
                    image_id, features = result
                    current_conf_features[image_id].update(features)


        # Convert the map for this confidence level into a DataFrame
        data_rows = []
        for img_id, features in current_conf_features.items():
            row = {'image_name': img_id}
            row.update(features)
            data_rows.append(row)

        df = pd.DataFrame(data_rows).set_index('image_name')

        # Define all expected columns for consistent zero-filling
        seg_cols_full = [f"seg_{cls}" for cls in SEG_FULL_CLASSES]
        obj_cols_full = [f"obj_{cls}" for cls in OBJ_FULL_CLASSES]
        all_cols_full = seg_cols_full + obj_cols_full

        # Ensure all possible columns exist and fill NaN (from missing entries) with 0.0
        df = df.reindex(columns=all_cols_full, fill_value=0.0)

        all_dataframes[f'conf_{conf_str}'] = df
        print(f"DataFrame for confidence {conf_str} created. Shape: {df.shape}")

    return all_dataframes


# --- Step 2: Process Feature Selection ---

def process_feature_selection(data_frames: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Filters the dataframes to only keep the most relevant, pre-selected classes."""
    print("\n--- Step 2: Feature Selection ---")

    # Now using ALL available classes as features
    selected_seg_cols = [f"seg_{cls}" for cls in SELECTED_SEG_CLASSES]
    selected_obj_cols = [f"obj_{cls}" for cls in SELECTED_OBJ_CLASSES]
    KEEP_COLS = selected_seg_cols + selected_obj_cols

    processed_dfs = {}
    for name, df in data_frames.items():
        filtered_df = df[KEEP_COLS].copy()
        processed_dfs[name] = filtered_df
        print(f"DataFrame {name} filtered. Final Feature Count: {filtered_df.shape[1]}")

    return processed_dfs

# --- Main Execution Block ---

def main():
    try:
        # Step 1: Download and Combine Data
        all_combined_dfs = download_and_combine_data()

        # Step 2: Feature Selection
        selected_dfs = process_feature_selection(all_combined_dfs)

        # Save the processed dataframes locally (e.g., in Colab's file system)
        for name, df in selected_dfs.items():
            save_path = f"processed_features_{name}.csv"
            df.to_csv(save_path)
            print(f"\nFinal feature file saved: {save_path} (Shape: {df.shape})")

    except FileNotFoundError as e:
        print(f"\nFATAL ERROR: {e}")
        print("Check the configured image features and Azure input paths.")
    except ConnectionError as e:
        print(f"\nFATAL ERROR: {e}")
        print("Please check your Azure CONNECTION_STRING and ensure the required libraries are installed.")
    except ValueError as e:
        print(f"\nFATAL ERROR: {e}")
        print("You must update the Azure CONNECTION_STRING placeholder.")
    except Exception as e:
        print(f"\nAn unexpected error occurred during the pipeline execution: {e}")

main()

In [ ]:
# for probabilities and all thresholds
import pandas as pd
import numpy as np
import json
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import io
from typing import Dict, Any, List
from concurrent.futures import ThreadPoolExecutor

import cv2
from skimage import color, filters

try:
    from azure.storage.blob import BlobServiceClient, ContainerClient
    from io import BytesIO
except ImportError:
    class BlobServiceClient: pass
    class ContainerClient: pass
    class BytesIO: pass
    print("Warning: azure-storage-blob is not installed. Azure download logic will not execute until installed and credentials are set.")


# --- Project Configuration ---

CONNECTION_STRING = AZURE_CONNECTION_STRING
CONTAINER_NAME = "london"
SEGMENTATION_FOLDER = "SemanticSegmentation/json/" # Ensure trailing slash
DETECTION_ROOT = "YoloV10/"
CONFIDENCE_LEVELS = [0.5]

# Placeholder for the complete set of Image IDs (19995 total)
NUM_IMAGES = 19995
IMAGE_IDS = [f"{i}_{angle}.jpg" for i in range(NUM_IMAGES // 4 + 1) for angle in [0, 90, 180, 270]][:NUM_IMAGES]

SEG_FULL_CLASSES = [
    'Banner', 'Barrier', 'Bench', 'Bicycle', 'Bicyclist',
    'Bike Lane', 'Bike Rack', 'Billboard', 'Bird', 'Boat',
    'Bridge', 'Building', 'Bus', 'CCTV Camera', 'Car',
    'Car Mount', 'Caravan', 'Catch Basin', 'Crosswalk - Plain',
    'Curb', 'Curb Cut', 'Ego Vehicle', 'Fence', 'Fire Hydrant',
    'Ground Animal', 'Guard Rail', 'Junction Box',
    'Lane Marking - Crosswalk', 'Lane Marking - General',
    'Mailbox', 'Manhole', 'Motorcycle', 'Motorcyclist',
    'Mountain', 'On Rails', 'Other Rider', 'Other Vehicle',
    'Parking', 'Pedestrian Area', 'Person', 'Phone Booth',
    'Pole', 'Pothole', 'Rail Track', 'Road', 'Sand',
    'Service Lane', 'Sidewalk', 'Sky', 'Snow', 'Street Light',
    'Terrain', 'Traffic Light', 'Traffic Sign (Back)',
    'Traffic Sign (Front)', 'Traffic Sign Frame', 'Trailer',
    'Trash Can', 'Truck', 'Tunnel', 'Utility Pole', 'Vegetation',
    'Wall', 'Water', 'Wheeled Slow'
]
OBJ_FULL_CLASSES = [
    'Person', 'Bicycle', 'Car', 'Motorcycle', 'Airplane', 'Bus', 'Train', 'Truck', 'Boat', 'Traffic Light',
    'Fire Hydrant', 'Stop Sign', 'Parking Meter', 'Bench', 'Bird', 'Cat', 'Dog', 'Horse', 'Sheep', 'Cow',
    'Elephant', 'Bear', 'Zebra', 'Giraffe', 'Backpack', 'Umbrella', 'Handbag', 'Tie', 'Suitcase', 'Frisbee',
    'Skis', 'Snowboard', 'Sports Ball', 'Kite', 'Baseball Bat', 'Baseball Glove', 'Skateboard', 'Surfboard',
    'Tennis Racket', 'Bottle', 'Wine Glass', 'Cup', 'Fork', 'Knife', 'Spoon', 'Bowl', 'Banana', 'Apple',
    'Sandwich', 'Orange', 'Broccoli', 'Carrot', 'Hot Dog', 'Pizza', 'Donut', 'Cake', 'Chair', 'Couch',
    'Potted Plant', 'Bed', 'Dining Table', 'Toilet', 'Tv', 'Laptop', 'Mouse', 'Remote', 'Keyboard',
    'Cell Phone', 'Microwave', 'Oven', 'Toaster', 'Sink', 'Refrigerator', 'Book', 'Clock', 'Vase',
    'Scissors', 'Teddy Bear', 'Hair Drier', 'Toothbrush'
]

# --- Step 2 Required Class Selection ---
SELECTED_SEG_CLASSES = SEG_FULL_CLASSES
SELECTED_OBJ_CLASSES = OBJ_FULL_CLASSES

# --- Helper Functions for Parsing --

def parse_image_stats(json_content: str) -> Dict[str, float]:
    """Parse image statistics JSON."""
    try:
        data = json.loads(json_content)

        # remove metadata fields
        data.pop("_source_blob", None)

        return data
    except Exception as e:
        print(f"Error parsing image stats JSON: {e}")
        return {}


def parse_segmentation(json_content: str) -> Dict[str, float]:
    """Parses segmentation JSON content and returns a dictionary of class names to pixel percentages."""
    try:
        data = json.loads(json_content)
        features = {}
        # Assuming a fixed image size like 640x640, or use the provided dimensions
        total_pixels = data.get('height', 640) * data.get('width', 640)

        for entry in data.get('label_histogram', []):
            # Store pixel percentage relative to the image size
            features[entry['label_name']] = entry['pixels'] / total_pixels
        return features
    except Exception as e:
        print(f"Error parsing segmentation JSON: {e}")
        return {}

def parse_yolo(txt_content: str, min_confidence: float = 0.5) -> Dict[str, float]:
    """Parses YOLO TXT content and returns a dictionary of object class summed confidences (probabilities)
    and summed bbox areas per class (if width/height are present)."""
    try:
        lines = txt_content.strip().split('\n')
        class_probs = {}
        for line in lines:
            parts = line.split()
            if not parts:
                continue
            # YOLO format assumed: class_id confidence x_center y_center width height
            class_id = int(parts[0])
            if len(parts) > 1:
                try:
                    conf = float(parts[1])
                except ValueError:
                    conf = 0.0
            else:
                conf = 0.0

            # apply a minimum confidence filter (safeguard)
            if conf < min_confidence:
                continue

            # map ID to class name
            if class_id < len(OBJ_FULL_CLASSES):
                class_name = OBJ_FULL_CLASSES[class_id]

                # sum confidences (existing behavior)
                class_probs[class_name] = class_probs.get(class_name, 0.0) + conf
                count_key = f"{class_name}_count"
                class_probs[count_key] = class_probs.get(count_key, 0.0) + 1.0

                # compute bbox area if width & height present (YOLO standard positions 4 and 5)
                if len(parts) >= 6:
                    try:
                        width = float(parts[4])
                        height = float(parts[5])
                        area = max(0.0, width * height)
                    except Exception:
                        area = 0.0
                    area_key = f"{class_name}_area"
                    class_probs[area_key] = class_probs.get(area_key, 0.0) + area

        return class_probs
    except Exception as e:
        print(f"Error parsing YOLO TXT: {e}")
        return {}

MAX_WORKERS = 32


def download_image_statistics(container_client):
    """Download image statistics JSON files and return a dataframe."""

    print("\n[Extra] Downloading Image Statistics...")

    blob_items = list(container_client.list_blobs(
        name_starts_with=IMAGE_STATS_FOLDER,
        results_per_page=1000
    ).by_page())

    blob_items = [blob for page in blob_items for blob in page]

    rows = []

    def process_blob(blob_item):
        try:
            blob_name = blob_item.name
            if not blob_name.endswith(".json"):
                return None

            blob_client = container_client.get_blob_client(blob_name)
            content = blob_client.download_blob().readall().decode("utf-8")

            stats = parse_image_stats(content)

            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            blob_base = blob_base.replace("_statistics", "")
            image_id = blob_base + ".jpg"

            stats["image_name"] = image_id
            return stats
        except:
            return None

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_blob, b) for b in blob_items]

        for f in tqdm(futures, desc="Downloading Image Stats"):
            r = f.result()
            if r:
                rows.append(r)

    df = pd.DataFrame(rows).set_index("image_name")

    return df

def download_and_process_blob(blob_item, container_client, is_segmentation: bool):
    """Downloads, reads, and parses a single blob item. Returns (image_id, features) or None on error."""
    try:
        blob_name = blob_item['name'] # Use the name from the blob dictionary/item

        blob_client = container_client.get_blob_client(blob_name)
        download_stream = blob_client.download_blob()
        content = download_stream.readall().decode('utf-8')

        if is_segmentation:
            if not blob_name.endswith('.json'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_segmentation(content)
            prefixed_features = {f"seg_{k}": v for k, v in features.items()}
        else:
            if not blob_name.endswith('.txt'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_yolo(content)
            prefixed_features = {f"obj_{k}": v for k, v in features.items()}

        if image_id not in IMAGE_IDS:
            return None

        return (image_id, prefixed_features)

    except Exception as e:
        return None

# --- Step 1: Download and Combine Logic ---

def download_and_combine_data() -> Dict[str, pd.DataFrame]:
    """
    Connects to Azure Blob Storage, downloads segmentation and object detection
    data for all confidence levels using parallel processing, and combines them.
    """
    print("--- Step 1: Connecting to Azure and Downloading All Blobs (Parallelized) ---")

    CONNECTION_STRING = ""

    try:
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    except Exception as e:
        raise ConnectionError(f"Failed to connect to Azure: {e}")

    # Initialize a master map to store all features for all images
    feature_map = {image_id: {} for image_id in IMAGE_IDS}

    # 1. Process Segmentation Data (Consistent across all confidence levels)
    print("\n[1/2] Processing Semantic Segmentation Data (JSON files)...")

    # Efficiently list all relevant segmentation blobs
    seg_blob_items = list(container_client.list_blobs(name_starts_with=SEGMENTATION_FOLDER, results_per_page=1000).by_page())
    seg_blob_items = [blob for page in seg_blob_items for blob in page] # Flatten list of pages

    seg_tasks = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for blob_item in seg_blob_items:
            seg_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, True))

        for future in tqdm(seg_tasks, desc="Downloading Segmentation (Parallel)"):
            result = future.result()
            if result:
                image_id, features = result
                feature_map[image_id].update(features)


    # 2. Process Object Detection Data for all 3 confidence levels
    all_dataframes = {}

    print("\n[2/2] Processing Object Detection (YOLO) Data (TXT files) for 3 Confidence Levels...")

    for conf in CONFIDENCE_LEVELS:
        conf_str = f"{conf:.1f}"
        detection_folder = f"{DETECTION_ROOT}conf_{conf_str}/labels/"

        # List all relevant detection blobs
        obj_blob_items = list(container_client.list_blobs(name_starts_with=detection_folder, results_per_page=1000).by_page())
        obj_blob_items = [blob for page in obj_blob_items for blob in page]

        current_conf_features = {img_id: feature_map[img_id].copy() for img_id in IMAGE_IDS}

        obj_tasks = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for blob_item in obj_blob_items:
                obj_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, False))

            for future in tqdm(obj_tasks, desc=f"Downloading YOLO {conf_str} (Parallel)"):
                result = future.result()
                if result:
                    image_id, features = result
                    current_conf_features[image_id].update(features)


        # Convert the map for this confidence level into a DataFrame
        data_rows = []
        for img_id, features in current_conf_features.items():
            row = {'image_name': img_id}
            row.update(features)
            data_rows.append(row)

        df = pd.DataFrame(data_rows).set_index('image_name')

        # Define all expected columns for consistent zero-filling
        seg_cols_full = [f"seg_{cls}" for cls in SEG_FULL_CLASSES]
        obj_cols_full = []
        for cls in OBJ_FULL_CLASSES:
            obj_cols_full.append(f"obj_{cls}")
            obj_cols_full.append(f"obj_{cls}_count")
            obj_cols_full.append(f"obj_{cls}_area")

        all_cols_full = seg_cols_full + obj_cols_full

        # Ensure all possible columns exist and fill NaN (from missing entries) with 0.0
        df = df.reindex(columns=all_cols_full, fill_value=0.0)

        all_dataframes[f'conf_{conf_str}'] = df
        print(f"DataFrame for confidence {conf_str} created. Shape: {df.shape}")

    return all_dataframes


# --- Step 2: Process Feature Selection ---

def process_feature_selection(data_frames: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Return the dataframes unchanged so all segmentation and object-detection columns are kept."""
    print("\n--- Step 2: Feature Selection (kept ALL columns) ---")

    processed_dfs = {}
    for name, df in data_frames.items():
        # keep everything as-is (make a copy to avoid side-effects)
        processed_dfs[name] = df.copy()
        print(f"DataFrame {name} kept all columns. Final Feature Count: {df.shape[1]}")

    return processed_dfs

IMAGE_STATS_FOLDER = "ImageStatistics/"

def extract_image_features(image_ids):
    rows = []

    for img_id in tqdm(image_ids, desc="Computing image features"):
        img_path = os.path.join(IMAGE_FOLDER, img_id)

        try:
            feats = compute_image_features(img_path)
            feats["image_name"] = img_id
            rows.append(feats)
        except Exception:
            # if image missing or error → fill zeros
            rows.append({"image_name": img_id})

    df = pd.DataFrame(rows).set_index("image_name")
    return df

def compute_image_features(image_path):
    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Image not found: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    features = {}

    ## ---------------- Luminance ----------------
    lum = img_gray.astype(np.float32)
    features['luminance_mean'] = np.mean(lum)
    features['luminance_std'] = np.std(lum)
    features['luminance_p05'] = np.percentile(lum, 5)
    features['luminance_p50'] = np.percentile(lum, 50)
    features['luminance_p95'] = np.percentile(lum, 95)
    features['dynamic_range'] = lum.max() - lum.min()
    features['frac_dark'] = np.mean(lum < 50)
    features['frac_bright'] = np.mean(lum > 205)

    ## ---------------- Color ----------------
    sat = img_hsv[:, :, 1].astype(np.float32)
    val = img_hsv[:, :, 2].astype(np.float32)
    features['sat_mean'] = np.mean(sat)
    features['sat_std'] = np.std(sat)
    features['val_mean'] = np.mean(val)
    features['val_std'] = np.std(val)

    # Colorfulness metric (Hasler & Süsstrunk)
    rg = img_rgb[:, :, 0].astype(np.float32) - img_rgb[:, :, 1].astype(np.float32)
    yb = 0.5 * (img_rgb[:, :, 0].astype(np.float32) + img_rgb[:, :, 1].astype(np.float32)) - img_rgb[:, :, 2].astype(np.float32)
    features['colorfulness'] = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3*np.sqrt(rg.mean()**2 + yb.mean()**2)

    # Warmth: Red vs Blue ratio
    R = img_rgb[:, :, 0].astype(np.float32)
    B = img_rgb[:, :, 2].astype(np.float32)
    features['warmth_rb'] = np.mean(R / (B + 1e-6))

    ## ---------------- Sharpness ----------------
    features['laplacian_var'] = cv2.Laplacian(img_gray, cv2.CV_64F).var()

    # Tenengrad (gradient magnitude)
    gx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = np.sqrt(gx**2 + gy**2)
    features['tenengrad_mean'] = np.mean(tenengrad)

    ## ---------------- Structure ----------------
    edges = cv2.Canny(img_gray, 100, 200)
    features['edge_density'] = np.mean(edges > 0)

    # Local contrast: std of small patches
    patch_size = 16
    patches = lum[:lum.shape[0]//patch_size*patch_size, :lum.shape[1]//patch_size*patch_size]
    patches = patches.reshape(patches.shape[0]//patch_size, patch_size,
                              patches.shape[1]//patch_size, patch_size)
    patches = patches.swapaxes(1,2).reshape(-1, patch_size, patch_size)
    local_contrast = [np.std(p) for p in patches]
    features['local_contrast_mean'] = np.mean(local_contrast)

    ## ---------------- Weather-ish ----------------
    # Dark channel prior
    min_channel = img_bgr.min(axis=2)
    features['dark_channel_mean'] = np.mean(min_channel)

    # Color attenuation (red - blue gradient)
    features['color_attenuation'] = np.mean(R - B)

    # White fraction
    features['white_fraction'] = np.mean(np.all(img_rgb > 240, axis=2))

    # Specular fraction: very bright pixels with low saturation
    features['specular_fraction'] = np.mean((val > 240) & (sat < 20))

    ## ---------------- Geometry ----------------
    # Vertical luminance gradient
    vert_grad = np.mean(np.diff(lum, axis=0), axis=1)
    features['vertical_lum_gradient'] = np.mean(vert_grad)

    return features

# --- Main Execution Block ---

def main():
    try:
        # Step 1: Download and Combine Data
        all_combined_dfs = download_and_combine_data()

        # Step 2: Feature Selection
        selected_dfs = process_feature_selection(all_combined_dfs)

        # NEW: compute image features
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)

        image_feature_df = download_image_statistics(container_client)

        # Merge with each dataframe
        for name, df in selected_dfs.items():

            df = df.join(image_feature_df, how="left").fillna(0)

            save_path = f"processed_features_{name}.csv"
            df.to_csv(save_path)

            print(f"\nFinal feature file saved: {save_path} (Shape: {df.shape})")

    except Exception as e:
        print(f"\nAn unexpected error occurred during the pipeline execution: {e}")

main()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv("enriched_collision_data_with_distance.csv")

seg_cols = [c for c in df.columns if c.startswith("seg_")]
df["seg_sum"] = df[seg_cols].sum(axis=1)

tol = 1e-6
bad = df[(df["seg_sum"] - 1).abs() > tol]

print("Total images:", len(df))
print("Images where seg_ sum ≈ 1:", len(df) - len(bad))
print("Images where seg_ sum != 1:", len(bad))

print("\nImages where seg_ columns do NOT sum to 1:")
print(bad[["latitude", "longitude","seg_sum"]])

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("processed_features_conf_0.5.csv")

# Drop two columns by name
df = df.drop(columns=["height", "width"])

# Save result
df.to_csv("processed_features_conf_0.5_probs.csv", index=False)

In [ ]:
school distance, distance overground, tube distance, highstreet distnace, pub_distance

In [ ]:
!pip install pandas pyproj scipy numpy
import pandas as pd
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree

def spatial_match_with_report(file1_path, file2_path, out_path, threshold_m=1.0):
    df1 = pd.read_csv(file1_path)
    df2 = pd.read_csv(file2_path)

    # ---- basic column checks ----
    if 'easting' not in df2.columns or 'northing' not in df2.columns:
        raise KeyError("file2 must contain 'easting' and 'northing' columns (BNG meters).")
    if 'latitude' not in df1.columns or 'longitude' not in df1.columns:
        raise KeyError("file1 must contain 'latitude' and 'longitude' columns (WGS84 degrees).")

    # ---- Convert df1 lat/lon -> BNG (meters) ----
    to_bng = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)
    lon1 = pd.to_numeric(df1['longitude'], errors='coerce').values
    lat1 = pd.to_numeric(df1['latitude'], errors='coerce').values

    valid1 = (~np.isnan(lon1)) & (~np.isnan(lat1))
    e1 = np.full(len(df1), np.nan)
    n1 = np.full(len(df1), np.nan)
    if valid1.any():
        e_valid, n_valid = to_bng.transform(lon1[valid1], lat1[valid1])
        e1[valid1] = e_valid
        n1[valid1] = n_valid

    # ---- Prepare df2 tree ----
    dist_cols = ["school distance", "distance overground", "tube distance", "highstreet distance", "pub_distance"]
    keep_cols = ['easting', 'northing'] + [c for c in dist_cols if c in df2.columns]
    df2_unique = df2[keep_cols].drop_duplicates(subset=['easting', 'northing']).reset_index(drop=True)

    e2 = pd.to_numeric(df2_unique['easting'], errors='coerce').values
    n2 = pd.to_numeric(df2_unique['northing'], errors='coerce').values
    valid2_mask = (~np.isnan(e2)) & (~np.isnan(n2))
    if not valid2_mask.all():
        df2_unique = df2_unique.loc[valid2_mask].reset_index(drop=True)
        e2 = e2[valid2_mask]
        n2 = n2[valid2_mask]

    tree = cKDTree(np.column_stack((e2, n2)))   # build tree in BNG meters

    # ---- Query (only for valid df1 rows) ----
    match_indices = np.full(len(df1), -1, dtype=int)
    match_dist_m = np.full(len(df1), np.nan)

    if valid1.any() and len(df2_unique) > 0:
        query_pts = np.column_stack((e1[valid1], n1[valid1]))
        # ********** MINIMAL CHANGE: remove n_jobs **********
        dists, idxs = tree.query(query_pts, k=1)  # returns distances in meters and indices
        match_indices[valid1] = idxs
        match_dist_m[valid1] = dists

    # ---- Reporting stats ----
    total = len(df1)
    valid_rows = valid1.sum()
    matched_nonneg = (match_indices >= 0)
    num_matched = int(matched_nonneg.sum())
    within_threshold = int(((match_dist_m <= threshold_m) & matched_nonneg).sum())
    pct_within = 100.0 * within_threshold / total if total else 0.0

    if num_matched > 0:
        max_dist = float(np.nanmax(match_dist_m[matched_nonneg]))
        mean_dist = float(np.nanmean(match_dist_m[matched_nonneg]))
        median_dist = float(np.nanmedian(match_dist_m[matched_nonneg]))
    else:
        max_dist = mean_dist = median_dist = float('nan')

    print(f"Total rows in file1: {total}")
    print(f"Rows with valid lat/lon: {valid_rows}")
    print(f"Nearest-neighbour results available: {num_matched}")
    print(f"Matches within {threshold_m} m: {within_threshold} ({pct_within:.2f}% of total rows)")
    print(f"Distances (meters) — max: {max_dist:.2f}, mean: {mean_dist:.2f}, median: {median_dist:.2f}")

    # ---- Attach matched distance columns to df1 ----
    for col in dist_cols:
        df1[col] = np.nan

    if num_matched > 0:
        matched_df2_rows = df2_unique.iloc[match_indices[matched_nonneg]].reset_index(drop=True)
        dst_idx = np.flatnonzero(matched_nonneg)
        for col in dist_cols:
            if col in df2_unique.columns:
                df1.loc[dst_idx, col] = matched_df2_rows[col].values

    df1['match_distance_m'] = match_dist_m
    df1.to_csv(out_path, index=False)
    print(f"Saved merged file to: {out_path}")
spatial_match_with_report("final_mapped_accidents.csv", "all vars_tested.csv", "final_mapped_accidents_dist_pub.csv")

In [ ]:
# grid

# for probabilities and all thresholds
import pandas as pd
import numpy as np
import json
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import io
from typing import Dict, Any, List
from concurrent.futures import ThreadPoolExecutor

import cv2
from skimage import color, filters

try:
    from azure.storage.blob import BlobServiceClient, ContainerClient
    from io import BytesIO
except ImportError:
    class BlobServiceClient: pass
    class ContainerClient: pass
    class BytesIO: pass
    print("Warning: azure-storage-blob is not installed. Azure download logic will not execute until installed and credentials are set.")


# --- Project Configuration ---

CONNECTION_STRING = AZURE_CONNECTION_STRING
CONTAINER_NAME = "london"
SEGMENTATION_FOLDER = "SemanticSegmentation/json/" # Ensure trailing slash
DETECTION_ROOT = "YoloV10/"
CONFIDENCE_LEVELS = [0.5]

# Placeholder for the complete set of Image IDs (19995 total)
NUM_IMAGES = 19995
IMAGE_IDS = [f"{i}_{angle}.jpg" for i in range(NUM_IMAGES // 4 + 1) for angle in [0, 90, 180, 270]][:NUM_IMAGES]

SEG_FULL_CLASSES = [
    'Banner', 'Barrier', 'Bench', 'Bicycle', 'Bicyclist',
    'Bike Lane', 'Bike Rack', 'Billboard', 'Bird', 'Boat',
    'Bridge', 'Building', 'Bus', 'CCTV Camera', 'Car',
    'Car Mount', 'Caravan', 'Catch Basin', 'Crosswalk - Plain',
    'Curb', 'Curb Cut', 'Ego Vehicle', 'Fence', 'Fire Hydrant',
    'Ground Animal', 'Guard Rail', 'Junction Box',
    'Lane Marking - Crosswalk', 'Lane Marking - General',
    'Mailbox', 'Manhole', 'Motorcycle', 'Motorcyclist',
    'Mountain', 'On Rails', 'Other Rider', 'Other Vehicle',
    'Parking', 'Pedestrian Area', 'Person', 'Phone Booth',
    'Pole', 'Pothole', 'Rail Track', 'Road', 'Sand',
    'Service Lane', 'Sidewalk', 'Sky', 'Snow', 'Street Light',
    'Terrain', 'Traffic Light', 'Traffic Sign (Back)',
    'Traffic Sign (Front)', 'Traffic Sign Frame', 'Trailer',
    'Trash Can', 'Truck', 'Tunnel', 'Utility Pole', 'Vegetation',
    'Wall', 'Water', 'Wheeled Slow'
]
OBJ_FULL_CLASSES = [
    'Person', 'Bicycle', 'Car', 'Motorcycle', 'Airplane', 'Bus', 'Train', 'Truck', 'Boat', 'Traffic Light',
    'Fire Hydrant', 'Stop Sign', 'Parking Meter', 'Bench', 'Bird', 'Cat', 'Dog', 'Horse', 'Sheep', 'Cow',
    'Elephant', 'Bear', 'Zebra', 'Giraffe', 'Backpack', 'Umbrella', 'Handbag', 'Tie', 'Suitcase', 'Frisbee',
    'Skis', 'Snowboard', 'Sports Ball', 'Kite', 'Baseball Bat', 'Baseball Glove', 'Skateboard', 'Surfboard',
    'Tennis Racket', 'Bottle', 'Wine Glass', 'Cup', 'Fork', 'Knife', 'Spoon', 'Bowl', 'Banana', 'Apple',
    'Sandwich', 'Orange', 'Broccoli', 'Carrot', 'Hot Dog', 'Pizza', 'Donut', 'Cake', 'Chair', 'Couch',
    'Potted Plant', 'Bed', 'Dining Table', 'Toilet', 'Tv', 'Laptop', 'Mouse', 'Remote', 'Keyboard',
    'Cell Phone', 'Microwave', 'Oven', 'Toaster', 'Sink', 'Refrigerator', 'Book', 'Clock', 'Vase',
    'Scissors', 'Teddy Bear', 'Hair Drier', 'Toothbrush'
]

# --- Step 2 Required Class Selection ---
SELECTED_SEG_CLASSES = SEG_FULL_CLASSES
SELECTED_OBJ_CLASSES = OBJ_FULL_CLASSES

# --- Helper Functions for Parsing --

def parse_image_stats(json_content: str) -> Dict[str, float]:
    """Parse image statistics JSON."""
    try:
        data = json.loads(json_content)

        # remove metadata fields
        data.pop("_source_blob", None)

        return data
    except Exception as e:
        print(f"Error parsing image stats JSON: {e}")
        return {}


def parse_segmentation(json_content: str) -> Dict[str, float]:
    """Parses segmentation JSON content and returns a dictionary of class names to pixel percentages."""
    try:
        data = json.loads(json_content)
        features = {}
        # Assuming a fixed image size like 640x640, or use the provided dimensions
        total_pixels = data.get('height', 640) * data.get('width', 640)

        for entry in data.get('label_histogram', []):
            # Store pixel percentage relative to the image size
            features[entry['label_name']] = entry['pixels'] / total_pixels
        return features
    except Exception as e:
        print(f"Error parsing segmentation JSON: {e}")
        return {}

def _grid_suffix_from_center(x_center: float, y_center: float, img_w: int, img_h: int) -> str:
    col = min(2, int(x_center / (img_w / 3))) + 1
    row = min(2, int(y_center / (img_h / 3))) + 1
    return f"_{row}_{col}"


def parse_yolo(txt_content: str, min_confidence: float = 0.5, img_w: int = 640, img_h: int = 640) -> Dict[str, float]:
    try:
        lines = txt_content.strip().split('\n')
        class_probs = {}
        for line in lines:
            parts = line.split()
            if not parts:
                continue

            class_id = int(parts[0])
            conf = float(parts[1]) if len(parts) > 1 else 0.0

            if conf < min_confidence:
                continue

            if class_id < len(OBJ_FULL_CLASSES):
                class_name = OBJ_FULL_CLASSES[class_id]

                grid_suffix = ""
                if len(parts) >= 4:
                    x_center = float(parts[2])
                    y_center = float(parts[3])
                    grid_suffix = _grid_suffix_from_center(x_center, y_center, img_w, img_h)

                conf_key = f"obj_{class_name}{grid_suffix}"
                class_probs[conf_key] = class_probs.get(conf_key, 0.0) + conf

                count_key = f"obj_{class_name}_count{grid_suffix}"
                class_probs[count_key] = class_probs.get(count_key, 0.0) + 1.0

                if len(parts) >= 6:
                    width = float(parts[4])
                    height = float(parts[5])
                    area = max(0.0, width * height)
                    area_key = f"obj_{class_name}_area{grid_suffix}"
                    class_probs[area_key] = class_probs.get(area_key, 0.0) + area

        return class_probs
    except Exception as e:
        print(f"Error parsing YOLO TXT: {e}")
        return {}

MAX_WORKERS = 32


def download_image_statistics(container_client):
    """Download image statistics JSON files and return a dataframe."""

    print("\n[Extra] Downloading Image Statistics...")

    blob_items = list(container_client.list_blobs(
        name_starts_with=IMAGE_STATS_FOLDER,
        results_per_page=1000
    ).by_page())

    blob_items = [blob for page in blob_items for blob in page]

    rows = []

    def process_blob(blob_item):
        try:
            blob_name = blob_item.name
            if not blob_name.endswith(".json"):
                return None

            blob_client = container_client.get_blob_client(blob_name)
            content = blob_client.download_blob().readall().decode("utf-8")

            stats = parse_image_stats(content)

            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            blob_base = blob_base.replace("_statistics", "")
            image_id = blob_base + ".jpg"

            stats["image_name"] = image_id
            return stats
        except:
            return None

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_blob, b) for b in blob_items]

        for f in tqdm(futures, desc="Downloading Image Stats"):
            r = f.result()
            if r:
                rows.append(r)

    df = pd.DataFrame(rows).set_index("image_name")

    return df

def download_and_process_blob(blob_item, container_client, is_segmentation: bool):
    """Downloads, reads, and parses a single blob item. Returns (image_id, features) or None on error."""
    try:
        blob_name = blob_item['name'] # Use the name from the blob dictionary/item

        blob_client = container_client.get_blob_client(blob_name)
        download_stream = blob_client.download_blob()
        content = download_stream.readall().decode('utf-8')

        if is_segmentation:
            if not blob_name.endswith('.json'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_segmentation(content)
            prefixed_features = {f"seg_{k}": v for k, v in features.items()}
        else:
            if not blob_name.endswith('.txt'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_yolo(content)
            prefixed_features = features

        if image_id not in IMAGE_IDS:
            return None

        return (image_id, prefixed_features)

    except Exception as e:
        return None

# --- Step 1: Download and Combine Logic ---

def download_and_combine_data() -> Dict[str, pd.DataFrame]:
    """
    Connects to Azure Blob Storage, downloads segmentation and object detection
    data for all confidence levels using parallel processing, and combines them.
    """
    print("--- Step 1: Connecting to Azure and Downloading All Blobs (Parallelized) ---")

    CONNECTION_STRING = ""

    try:
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    except Exception as e:
        raise ConnectionError(f"Failed to connect to Azure: {e}")

    # Initialize a master map to store all features for all images
    feature_map = {image_id: {} for image_id in IMAGE_IDS}

    # 1. Process Segmentation Data (Consistent across all confidence levels)
    print("\n[1/2] Processing Semantic Segmentation Data (JSON files)...")

    # Efficiently list all relevant segmentation blobs
    seg_blob_items = list(container_client.list_blobs(name_starts_with=SEGMENTATION_FOLDER, results_per_page=1000).by_page())
    seg_blob_items = [blob for page in seg_blob_items for blob in page] # Flatten list of pages

    seg_tasks = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for blob_item in seg_blob_items:
            seg_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, True))

        for future in tqdm(seg_tasks, desc="Downloading Segmentation (Parallel)"):
            result = future.result()
            if result:
                image_id, features = result
                feature_map[image_id].update(features)


    # 2. Process Object Detection Data for all 3 confidence levels
    all_dataframes = {}

    print("\n[2/2] Processing Object Detection (YOLO) Data (TXT files) for 3 Confidence Levels...")

    for conf in CONFIDENCE_LEVELS:
        conf_str = f"{conf:.1f}"
        detection_folder = f"{DETECTION_ROOT}conf_{conf_str}/labels/"

        # List all relevant detection blobs
        obj_blob_items = list(container_client.list_blobs(name_starts_with=detection_folder, results_per_page=1000).by_page())
        obj_blob_items = [blob for page in obj_blob_items for blob in page]

        current_conf_features = {img_id: feature_map[img_id].copy() for img_id in IMAGE_IDS}

        obj_tasks = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for blob_item in obj_blob_items:
                obj_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, False))

            for future in tqdm(obj_tasks, desc=f"Downloading YOLO {conf_str} (Parallel)"):
                result = future.result()
                if result:
                    image_id, features = result
                    current_conf_features[image_id].update(features)


        # Convert the map for this confidence level into a DataFrame
        data_rows = []
        for img_id, features in current_conf_features.items():
            row = {'image_name': img_id}
            row.update(features)
            data_rows.append(row)

        df = pd.DataFrame(data_rows).set_index('image_name')

        # Define all expected columns for consistent zero-filling
        seg_cols_full = [f"seg_{cls}" for cls in SEG_FULL_CLASSES]
        obj_cols_full = []
        for cls in OBJ_FULL_CLASSES:
            for r in range(1, 4):
                for c in range(1, 4):
                    suf = f"_{r}_{c}"
                    obj_cols_full.append(f"obj_{cls}{suf}")
                    obj_cols_full.append(f"obj_{cls}_count{suf}")
                    obj_cols_full.append(f"obj_{cls}_area{suf}")

        all_cols_full = seg_cols_full + obj_cols_full

        # Ensure all possible columns exist and fill NaN (from missing entries) with 0.0
        df = df.reindex(columns=all_cols_full, fill_value=0.0)

        all_dataframes[f'conf_{conf_str}'] = df
        print(f"DataFrame for confidence {conf_str} created. Shape: {df.shape}")

    return all_dataframes


# --- Step 2: Process Feature Selection ---

def process_feature_selection(data_frames: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Return the dataframes unchanged so all segmentation and object-detection columns are kept."""
    print("\n--- Step 2: Feature Selection (kept ALL columns) ---")

    processed_dfs = {}
    for name, df in data_frames.items():
        # keep everything as-is (make a copy to avoid side-effects)
        processed_dfs[name] = df.copy()
        print(f"DataFrame {name} kept all columns. Final Feature Count: {df.shape[1]}")

    return processed_dfs

IMAGE_STATS_FOLDER = "ImageStatistics/"

def extract_image_features(image_ids):
    rows = []

    for img_id in tqdm(image_ids, desc="Computing image features"):
        img_path = os.path.join(IMAGE_FOLDER, img_id)

        try:
            feats = compute_image_features(img_path)
            feats["image_name"] = img_id
            rows.append(feats)
        except Exception:
            # if image missing or error → fill zeros
            rows.append({"image_name": img_id})

    df = pd.DataFrame(rows).set_index("image_name")
    return df

def compute_image_features(image_path):
    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Image not found: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    features = {}

    ## ---------------- Luminance ----------------
    lum = img_gray.astype(np.float32)
    features['luminance_mean'] = np.mean(lum)
    features['luminance_std'] = np.std(lum)
    features['luminance_p05'] = np.percentile(lum, 5)
    features['luminance_p50'] = np.percentile(lum, 50)
    features['luminance_p95'] = np.percentile(lum, 95)
    features['dynamic_range'] = lum.max() - lum.min()
    features['frac_dark'] = np.mean(lum < 50)
    features['frac_bright'] = np.mean(lum > 205)

    ## ---------------- Color ----------------
    sat = img_hsv[:, :, 1].astype(np.float32)
    val = img_hsv[:, :, 2].astype(np.float32)
    features['sat_mean'] = np.mean(sat)
    features['sat_std'] = np.std(sat)
    features['val_mean'] = np.mean(val)
    features['val_std'] = np.std(val)

    # Colorfulness metric (Hasler & Süsstrunk)
    rg = img_rgb[:, :, 0].astype(np.float32) - img_rgb[:, :, 1].astype(np.float32)
    yb = 0.5 * (img_rgb[:, :, 0].astype(np.float32) + img_rgb[:, :, 1].astype(np.float32)) - img_rgb[:, :, 2].astype(np.float32)
    features['colorfulness'] = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3*np.sqrt(rg.mean()**2 + yb.mean()**2)

    # Warmth: Red vs Blue ratio
    R = img_rgb[:, :, 0].astype(np.float32)
    B = img_rgb[:, :, 2].astype(np.float32)
    features['warmth_rb'] = np.mean(R / (B + 1e-6))

    ## ---------------- Sharpness ----------------
    features['laplacian_var'] = cv2.Laplacian(img_gray, cv2.CV_64F).var()

    # Tenengrad (gradient magnitude)
    gx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = np.sqrt(gx**2 + gy**2)
    features['tenengrad_mean'] = np.mean(tenengrad)

    ## ---------------- Structure ----------------
    edges = cv2.Canny(img_gray, 100, 200)
    features['edge_density'] = np.mean(edges > 0)

    # Local contrast: std of small patches
    patch_size = 16
    patches = lum[:lum.shape[0]//patch_size*patch_size, :lum.shape[1]//patch_size*patch_size]
    patches = patches.reshape(patches.shape[0]//patch_size, patch_size,
                              patches.shape[1]//patch_size, patch_size)
    patches = patches.swapaxes(1,2).reshape(-1, patch_size, patch_size)
    local_contrast = [np.std(p) for p in patches]
    features['local_contrast_mean'] = np.mean(local_contrast)

    ## ---------------- Weather-ish ----------------
    # Dark channel prior
    min_channel = img_bgr.min(axis=2)
    features['dark_channel_mean'] = np.mean(min_channel)

    # Color attenuation (red - blue gradient)
    features['color_attenuation'] = np.mean(R - B)

    # White fraction
    features['white_fraction'] = np.mean(np.all(img_rgb > 240, axis=2))

    # Specular fraction: very bright pixels with low saturation
    features['specular_fraction'] = np.mean((val > 240) & (sat < 20))

    ## ---------------- Geometry ----------------
    # Vertical luminance gradient
    vert_grad = np.mean(np.diff(lum, axis=0), axis=1)
    features['vertical_lum_gradient'] = np.mean(vert_grad)

    return features

# --- Main Execution Block ---

def main():
    try:
        # Step 1: Download and Combine Data
        all_combined_dfs = download_and_combine_data()

        # Step 2: Feature Selection
        selected_dfs = process_feature_selection(all_combined_dfs)

        # NEW: compute image features
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)

        image_feature_df = download_image_statistics(container_client)

        # Merge with each dataframe
        for name, df in selected_dfs.items():

            df = df.join(image_feature_df, how="left").fillna(0)

            save_path = f"processed_features_{name}.csv"
            df.to_csv(save_path)

            print(f"\nFinal feature file saved: {save_path} (Shape: {df.shape})")

    except Exception as e:
        print(f"\nAn unexpected error occurred during the pipeline execution: {e}")

main()

In [ ]:
# grid for semantic segmentation as well

# for probabilities and all thresholds
import pandas as pd
import numpy as np
import json
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import io
from typing import Dict, Any, List
from concurrent.futures import ThreadPoolExecutor

import cv2
from skimage import color, filters

try:
    from azure.storage.blob import BlobServiceClient, ContainerClient
    from io import BytesIO
except ImportError:
    class BlobServiceClient: pass
    class ContainerClient: pass
    class BytesIO: pass
    print("Warning: azure-storage-blob is not installed. Azure download logic will not execute until installed and credentials are set.")


# --- Project Configuration ---

CONNECTION_STRING = AZURE_CONNECTION_STRING
CONTAINER_NAME = "london"
SEGMENTATION_FOLDER = "SemanticSegmentation/json/" # Ensure trailing slash
SEGMENTATION_OVERLAY_FOLDER = "SemanticSegmentation/overlays/"
DETECTION_ROOT = "YoloV10/"
CONFIDENCE_LEVELS = [0.5]

# Placeholder for the complete set of Image IDs (19995 total)
NUM_IMAGES = 19995
IMAGE_IDS = [f"{i}_{angle}.jpg" for i in range(NUM_IMAGES // 4 + 1) for angle in [0, 90, 180, 270]][:NUM_IMAGES]

SEG_FULL_CLASSES = [
    'Banner', 'Barrier', 'Bench', 'Bicycle', 'Bicyclist',
    'Bike Lane', 'Bike Rack', 'Billboard', 'Bird', 'Boat',
    'Bridge', 'Building', 'Bus', 'CCTV Camera', 'Car',
    'Car Mount', 'Caravan', 'Catch Basin', 'Crosswalk - Plain',
    'Curb', 'Curb Cut', 'Ego Vehicle', 'Fence', 'Fire Hydrant',
    'Ground Animal', 'Guard Rail', 'Junction Box',
    'Lane Marking - Crosswalk', 'Lane Marking - General',
    'Mailbox', 'Manhole', 'Motorcycle', 'Motorcyclist',
    'Mountain', 'On Rails', 'Other Rider', 'Other Vehicle',
    'Parking', 'Pedestrian Area', 'Person', 'Phone Booth',
    'Pole', 'Pothole', 'Rail Track', 'Road', 'Sand',
    'Service Lane', 'Sidewalk', 'Sky', 'Snow', 'Street Light',
    'Terrain', 'Traffic Light', 'Traffic Sign (Back)',
    'Traffic Sign (Front)', 'Traffic Sign Frame', 'Trailer',
    'Trash Can', 'Truck', 'Tunnel', 'Utility Pole', 'Vegetation',
    'Wall', 'Water', 'Wheeled Slow'
]
OBJ_FULL_CLASSES = [
    'Person', 'Bicycle', 'Car', 'Motorcycle', 'Airplane', 'Bus', 'Train', 'Truck', 'Boat', 'Traffic Light',
    'Fire Hydrant', 'Stop Sign', 'Parking Meter', 'Bench', 'Bird', 'Cat', 'Dog', 'Horse', 'Sheep', 'Cow',
    'Elephant', 'Bear', 'Zebra', 'Giraffe', 'Backpack', 'Umbrella', 'Handbag', 'Tie', 'Suitcase', 'Frisbee',
    'Skis', 'Snowboard', 'Sports Ball', 'Kite', 'Baseball Bat', 'Baseball Glove', 'Skateboard', 'Surfboard',
    'Tennis Racket', 'Bottle', 'Wine Glass', 'Cup', 'Fork', 'Knife', 'Spoon', 'Bowl', 'Banana', 'Apple',
    'Sandwich', 'Orange', 'Broccoli', 'Carrot', 'Hot Dog', 'Pizza', 'Donut', 'Cake', 'Chair', 'Couch',
    'Potted Plant', 'Bed', 'Dining Table', 'Toilet', 'Tv', 'Laptop', 'Mouse', 'Remote', 'Keyboard',
    'Cell Phone', 'Microwave', 'Oven', 'Toaster', 'Sink', 'Refrigerator', 'Book', 'Clock', 'Vase',
    'Scissors', 'Teddy Bear', 'Hair Drier', 'Toothbrush'
]

# --- Step 2 Required Class Selection ---
SELECTED_SEG_CLASSES = SEG_FULL_CLASSES
SELECTED_OBJ_CLASSES = OBJ_FULL_CLASSES

# --- Helper Functions for Parsing --

def parse_image_stats(json_content: str) -> Dict[str, float]:
    """Parse image statistics JSON."""
    try:
        data = json.loads(json_content)

        # remove metadata fields
        data.pop("_source_blob", None)

        return data
    except Exception as e:
        print(f"Error parsing image stats JSON: {e}")
        return {}


def parse_segmentation(json_content: str) -> Dict[str, float]:
    """Parses segmentation JSON content and returns a dictionary of class names to pixel percentages."""
    try:
        data = json.loads(json_content)
        features = {}
        # Assuming a fixed image size like 640x640, or use the provided dimensions
        total_pixels = data.get('height', 640) * data.get('width', 640)

        for entry in data.get('label_histogram', []):
            # Store pixel percentage relative to the image size
            features[entry['label_name']] = entry['pixels'] / total_pixels
        return features
    except Exception as e:
        print(f"Error parsing segmentation JSON: {e}")
        return {}

def parse_segmentation_grid_overlay(png_bytes: bytes, json_content: str) -> Dict[str, float]:
    """
    Count pixels per class per 3x3 grid cell from the colorful segmentation PNG.
    Assumes the PNG is a true color mask / visualization with consistent colors.
    """
    try:
        data = json.loads(json_content)

        img = cv2.imdecode(np.frombuffer(png_bytes, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            return {}

        h, w = img.shape[:2]
        x_bins = np.array_split(np.arange(w), 3)
        y_bins = np.array_split(np.arange(h), 3)

        colors, counts = np.unique(img.reshape(-1, 3), axis=0, return_counts=True)

        color_counts = {tuple(c): cnt for c, cnt in zip(colors, counts)}
        label_counts = {e["label_name"]: e["pixels"] for e in data["label_histogram"]}

        color_to_label = {}
        used = set()

        for color, cnt in color_counts.items():
            best_label = min(
                (l for l in label_counts if l not in used),
                key=lambda l: abs(label_counts[l] - cnt)
            )
            color_to_label[color] = best_label
            used.add(best_label)

        feats = {}
        for r, ys in enumerate(y_bins, start=1):
            for c, xs in enumerate(x_bins, start=1):
                cell = img[np.ix_(ys, xs)].reshape(-1, 3)
                for color, label in color_to_label.items():
                    px = int(np.sum(np.all(cell == color, axis=1)))
                    if px:
                        cell_area = len(ys) * len(xs)
                        feats[f"{label}_{r}_{c}"] = px / cell_area

        # ---------------- GLOBAL CLASS CENTROIDS ----------------

        for color, label in color_to_label.items():

            mask = np.all(img == np.array(color), axis=2)

            ys_all, xs_all = np.where(mask)

            if len(xs_all) == 0:
                continue

            # weighted center of mass
            cx = xs_all.mean() / w
            cy = ys_all.mean() / h

            feats[f"{label}_cx"] = cx
            feats[f"{label}_cy"] = cy

            mask_u8 = mask.astype(np.uint8)
            contours, _ = cv2.findContours(
                mask_u8,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )
            perimeter = sum(cv2.arcLength(cnt, True) for cnt in contours)

            perimeter /= (2 * (w + h))  # normalized
            feats[f"{label}_perimeter"] = perimeter

        return feats
    except Exception as e:
        print(f"Error parsing segmentation grid overlay: {e}")
        return {}

def _grid_suffix_from_center(x_center: float, y_center: float, img_w: int, img_h: int) -> str:
    col = min(2, int(x_center / (img_w / 3))) + 1
    row = min(2, int(y_center / (img_h / 3))) + 1
    return f"_{row}_{col}"


def parse_yolo(txt_content: str, min_confidence: float = 0.5, img_w: int = 640, img_h: int = 640) -> Dict[str, float]:
    try:
        lines = txt_content.strip().split('\n')
        class_probs = {}
        for line in lines:
            parts = line.split()
            if not parts:
                continue

            class_id = int(parts[0])
            conf = float(parts[1]) if len(parts) > 1 else 0.0

            if conf < min_confidence:
                continue

            if class_id < len(OBJ_FULL_CLASSES):
                class_name = OBJ_FULL_CLASSES[class_id]

                grid_suffix = ""
                if len(parts) >= 4:
                    x_center = float(parts[2])
                    y_center = float(parts[3])
                    grid_suffix = _grid_suffix_from_center(x_center, y_center, img_w, img_h)

                conf_key = f"obj_{class_name}{grid_suffix}"
                class_probs[conf_key] = class_probs.get(conf_key, 0.0) + conf

                count_key = f"obj_{class_name}_count{grid_suffix}"
                class_probs[count_key] = class_probs.get(count_key, 0.0) + 1.0

                if len(parts) >= 6:
                    width = float(parts[4])
                    height = float(parts[5])
                    area = max(0.0, width * height)
                    area_key = f"obj_{class_name}_area{grid_suffix}"
                    class_probs[area_key] = class_probs.get(area_key, 0.0) + area

        return class_probs
    except Exception as e:
        print(f"Error parsing YOLO TXT: {e}")
        return {}

MAX_WORKERS = 32


def download_image_statistics(container_client):
    """Download image statistics JSON files and return a dataframe."""

    print("\n[Extra] Downloading Image Statistics...")

    blob_items = list(container_client.list_blobs(
        name_starts_with=IMAGE_STATS_FOLDER,
        results_per_page=1000
    ).by_page())

    blob_items = [blob for page in blob_items for blob in page]

    rows = []

    def process_blob(blob_item):
        try:
            blob_name = blob_item.name
            if not blob_name.endswith(".json"):
                return None

            blob_client = container_client.get_blob_client(blob_name)
            content = blob_client.download_blob().readall().decode("utf-8")

            stats = parse_image_stats(content)

            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            blob_base = blob_base.replace("_statistics", "")
            image_id = blob_base + ".jpg"

            stats["image_name"] = image_id
            return stats
        except:
            return None

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_blob, b) for b in blob_items]

        for f in tqdm(futures, desc="Downloading Image Stats"):
            r = f.result()
            if r:
                rows.append(r)

    df = pd.DataFrame(rows).set_index("image_name")

    return df

def download_and_process_blob(blob_item, container_client, is_segmentation: bool):
    """Downloads, reads, and parses a single blob item. Returns (image_id, features) or None on error."""
    try:
        blob_name = blob_item['name'] # Use the name from the blob dictionary/item

        blob_client = container_client.get_blob_client(blob_name)
        download_stream = blob_client.download_blob()
        content = download_stream.readall().decode('utf-8')

        if is_segmentation:
          if not blob_name.endswith(".json"):
              return None

          blob_base = os.path.splitext(os.path.basename(blob_name))[0]
          image_id = blob_base + ".jpg"

          # global semantic segmentation features
          features = parse_segmentation(content)

          # IMPORTANT: Azure blob names should use forward slashes, not os.path.join
          overlay_blob_name = f"{SEGMENTATION_OVERLAY_FOLDER}{blob_base}_overlay.png"

          try:
              overlay_bytes = container_client.get_blob_client(overlay_blob_name).download_blob().readall()
              grid_features = parse_segmentation_grid_overlay(overlay_bytes, content)
              features.update(grid_features)
          except Exception as e:
              print(f"[WARN] overlay/grid missing for {blob_base}: {e}")

          prefixed_features = {f"seg_{k}": v for k, v in features.items()}
        else:
            if not blob_name.endswith('.txt'): return None
            blob_base = os.path.splitext(os.path.basename(blob_name))[0]
            image_id = blob_base + '.jpg'
            features = parse_yolo(content)
            prefixed_features = features

        if image_id not in IMAGE_IDS:
            return None

        return (image_id, prefixed_features)

    except Exception as e:
        return None

# --- Step 1: Download and Combine Logic ---

def download_and_combine_data() -> Dict[str, pd.DataFrame]:
    """
    Connects to Azure Blob Storage, downloads segmentation and object detection
    data for all confidence levels using parallel processing, and combines them.
    """
    print("--- Step 1: Connecting to Azure and Downloading All Blobs (Parallelized) ---")

    CONNECTION_STRING = ""

    try:
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    except Exception as e:
        raise ConnectionError(f"Failed to connect to Azure: {e}")

    # Initialize a master map to store all features for all images
    feature_map = {image_id: {} for image_id in IMAGE_IDS}

    # 1. Process Segmentation Data (Consistent across all confidence levels)
    print("\n[1/2] Processing Semantic Segmentation Data (JSON files)...")

    # Efficiently list all relevant segmentation blobs
    seg_blob_items = list(container_client.list_blobs(name_starts_with=SEGMENTATION_FOLDER, results_per_page=1000).by_page())
    seg_blob_items = [blob for page in seg_blob_items for blob in page] # Flatten list of pages

    seg_tasks = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for blob_item in seg_blob_items:
            seg_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, True))

        for future in tqdm(seg_tasks, desc="Downloading Segmentation (Parallel)"):
            result = future.result()
            if result:
                image_id, features = result
                feature_map[image_id].update(features)


    # 2. Process Object Detection Data for all 3 confidence levels
    all_dataframes = {}

    print("\n[2/2] Processing Object Detection (YOLO) Data (TXT files) for 3 Confidence Levels...")

    for conf in CONFIDENCE_LEVELS:
        conf_str = f"{conf:.1f}"
        detection_folder = f"{DETECTION_ROOT}conf_{conf_str}/labels/"

        # List all relevant detection blobs
        obj_blob_items = list(container_client.list_blobs(name_starts_with=detection_folder, results_per_page=1000).by_page())
        obj_blob_items = [blob for page in obj_blob_items for blob in page]

        current_conf_features = {img_id: feature_map[img_id].copy() for img_id in IMAGE_IDS}

        obj_tasks = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for blob_item in obj_blob_items:
                obj_tasks.append(executor.submit(download_and_process_blob, blob_item, container_client, False))

            for future in tqdm(obj_tasks, desc=f"Downloading YOLO {conf_str} (Parallel)"):
                result = future.result()
                if result:
                    image_id, features = result
                    current_conf_features[image_id].update(features)


        # Convert the map for this confidence level into a DataFrame
        data_rows = []
        for img_id, features in current_conf_features.items():
            row = {'image_name': img_id}
            row.update(features)
            data_rows.append(row)

        df = pd.DataFrame(data_rows).set_index('image_name')

        # Define all expected columns for consistent zero-filling
        seg_cols_full = [f"seg_{cls}" for cls in SEG_FULL_CLASSES]
        seg_cols_full += [
            f"seg_{cls}_{r}_{c}"
            for cls in SEG_FULL_CLASSES
            for r in range(1, 4)
            for c in range(1, 4)
        ]
        seg_cols_full += [
            f"seg_{cls}_cx"
            for cls in SEG_FULL_CLASSES
        ]

        seg_cols_full += [
            f"seg_{cls}_cy"
            for cls in SEG_FULL_CLASSES
        ]
        seg_cols_full += [f"seg_{cls}_perimeter" for cls in SEG_FULL_CLASSES]
        obj_cols_full = []
        for cls in OBJ_FULL_CLASSES:
            for r in range(1, 4):
                for c in range(1, 4):
                    suf = f"_{r}_{c}"
                    obj_cols_full.append(f"obj_{cls}{suf}")
                    obj_cols_full.append(f"obj_{cls}_count{suf}")
                    obj_cols_full.append(f"obj_{cls}_area{suf}")

        all_cols_full = seg_cols_full + obj_cols_full

        # Ensure all possible columns exist and fill NaN (from missing entries) with 0.0
        df = df.reindex(columns=all_cols_full, fill_value=0.0)

        all_dataframes[f'conf_{conf_str}'] = df
        print(f"DataFrame for confidence {conf_str} created. Shape: {df.shape}")

    return all_dataframes


# --- Step 2: Process Feature Selection ---

def process_feature_selection(data_frames: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Return the dataframes unchanged so all segmentation and object-detection columns are kept."""
    print("\n--- Step 2: Feature Selection (kept ALL columns) ---")

    processed_dfs = {}
    for name, df in data_frames.items():
        # keep everything as-is (make a copy to avoid side-effects)
        processed_dfs[name] = df.copy()
        print(f"DataFrame {name} kept all columns. Final Feature Count: {df.shape[1]}")

    return processed_dfs

IMAGE_STATS_FOLDER = "ImageStatistics/"

def extract_image_features(image_ids):
    rows = []

    for img_id in tqdm(image_ids, desc="Computing image features"):
        img_path = os.path.join(IMAGE_FOLDER, img_id)

        try:
            feats = compute_image_features(img_path)
            feats["image_name"] = img_id
            rows.append(feats)
        except Exception:
            # if image missing or error → fill zeros
            rows.append({"image_name": img_id})

    df = pd.DataFrame(rows).set_index("image_name")
    return df

def compute_image_features(image_path):
    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Image not found: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    features = {}

    ## ---------------- Luminance ----------------
    lum = img_gray.astype(np.float32)
    features['luminance_mean'] = np.mean(lum)
    features['luminance_std'] = np.std(lum)
    features['luminance_p05'] = np.percentile(lum, 5)
    features['luminance_p50'] = np.percentile(lum, 50)
    features['luminance_p95'] = np.percentile(lum, 95)
    features['dynamic_range'] = lum.max() - lum.min()
    features['frac_dark'] = np.mean(lum < 50)
    features['frac_bright'] = np.mean(lum > 205)

    ## ---------------- Color ----------------
    sat = img_hsv[:, :, 1].astype(np.float32)
    val = img_hsv[:, :, 2].astype(np.float32)
    features['sat_mean'] = np.mean(sat)
    features['sat_std'] = np.std(sat)
    features['val_mean'] = np.mean(val)
    features['val_std'] = np.std(val)

    # Colorfulness metric (Hasler & Süsstrunk)
    rg = img_rgb[:, :, 0].astype(np.float32) - img_rgb[:, :, 1].astype(np.float32)
    yb = 0.5 * (img_rgb[:, :, 0].astype(np.float32) + img_rgb[:, :, 1].astype(np.float32)) - img_rgb[:, :, 2].astype(np.float32)
    features['colorfulness'] = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3*np.sqrt(rg.mean()**2 + yb.mean()**2)

    # Warmth: Red vs Blue ratio
    R = img_rgb[:, :, 0].astype(np.float32)
    B = img_rgb[:, :, 2].astype(np.float32)
    features['warmth_rb'] = np.mean(R / (B + 1e-6))

    ## ---------------- Sharpness ----------------
    features['laplacian_var'] = cv2.Laplacian(img_gray, cv2.CV_64F).var()

    # Tenengrad (gradient magnitude)
    gx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = np.sqrt(gx**2 + gy**2)
    features['tenengrad_mean'] = np.mean(tenengrad)

    ## ---------------- Structure ----------------
    edges = cv2.Canny(img_gray, 100, 200)
    features['edge_density'] = np.mean(edges > 0)

    # Local contrast: std of small patches
    patch_size = 16
    patches = lum[:lum.shape[0]//patch_size*patch_size, :lum.shape[1]//patch_size*patch_size]
    patches = patches.reshape(patches.shape[0]//patch_size, patch_size,
                              patches.shape[1]//patch_size, patch_size)
    patches = patches.swapaxes(1,2).reshape(-1, patch_size, patch_size)
    local_contrast = [np.std(p) for p in patches]
    features['local_contrast_mean'] = np.mean(local_contrast)

    ## ---------------- Weather-ish ----------------
    # Dark channel prior
    min_channel = img_bgr.min(axis=2)
    features['dark_channel_mean'] = np.mean(min_channel)

    # Color attenuation (red - blue gradient)
    features['color_attenuation'] = np.mean(R - B)

    # White fraction
    features['white_fraction'] = np.mean(np.all(img_rgb > 240, axis=2))

    # Specular fraction: very bright pixels with low saturation
    features['specular_fraction'] = np.mean((val > 240) & (sat < 20))

    ## ---------------- Geometry ----------------
    # Vertical luminance gradient
    vert_grad = np.mean(np.diff(lum, axis=0), axis=1)
    features['vertical_lum_gradient'] = np.mean(vert_grad)

    return features

# --- Main Execution Block ---

def main():
    try:
        # Step 1: Download and Combine Data
        all_combined_dfs = download_and_combine_data()

        # Step 2: Feature Selection
        selected_dfs = process_feature_selection(all_combined_dfs)

        # NEW: compute image features
        blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)

        # image_feature_df = download_image_statistics(container_client)

        # Merge with each dataframe
        for name, df in selected_dfs.items():

            # df = df.join(image_feature_df, how="left").fillna(0)

            save_path = f"processed_features_{name}.csv"
            df.to_csv(save_path)

            print(f"\nFinal feature file saved: {save_path} (Shape: {df.shape})")

    except Exception as e:
        print(f"\nAn unexpected error occurred during the pipeline execution: {e}")

main()

In [ ]:
import pandas as pd

# Input and output files
input_file = "processed_features_conf_0.5.csv"
output_file = "processed_features_conf_0.5_img_100_rows.csv"

# Load CSV
df = pd.read_csv(input_file)
df_subset = df.head(100)

# Save to new CSV
df_subset.to_csv(output_file, index=False)

print(f"Saved new file with shape: {df_subset.shape} -> {output_file}")

In [ ]:
seg building  -  0.00759521484375

In [ ]:
import pandas as pd
import numpy as np

# --- LOAD DATA ---
df = pd.read_csv("processed_features_conf_0.5_segmn_grid.csv", index_col="image_name")

# tolerance for floating error
TOL = 1e-3

print("\n=== SEGMENTATION GRID SANITY CHECK ===\n")

all_bad = 0

for r in range(1, 4):
    for c in range(1, 4):
        # get columns for this cell
        cols = [col for col in df.columns if col.startswith("seg_") and col.endswith(f"_{r}_{c}")]

        if len(cols) == 0:
            print(f"[WARNING] Cell {r}_{c}: no columns found")
            continue

        # sum across classes
        s = df[cols].sum(axis=1)

        # stats
        mean = s.mean()
        min_v = s.min()
        max_v = s.max()
        std = s.std()

        # bad rows
        bad_mask = (s < 1 - TOL) | (s > 1 + TOL)
        num_bad = bad_mask.sum()
        all_bad += num_bad

        print(f"Cell {r}_{c}:")
        print(f"  cols: {len(cols)}")
        print(f"  mean: {mean:.6f}")
        print(f"  std : {std:.6f}")
        print(f"  min : {min_v:.6f}")
        print(f"  max : {max_v:.6f}")
        print(f"  bad : {num_bad}")

        # show worst offenders (only if bad exists)
        if num_bad > 0:
            print("  worst low:")
            print(s.sort_values().head(3))
            print("  worst high:")
            print(s.sort_values().tail(3))

        print()

print("=== SUMMARY ===")
print(f"Total bad rows across all cells: {all_bad}")

In [ ]:
import pandas as pd

# Path to your CSV file
csv_file = "processed_features_conf_0.5.csv"

# Read the CSV
df = pd.read_csv(csv_file)

# Print headers
print("Headers:")
print(df.columns.tolist())

# Print first 10 rows
print("\nFirst 10 rows:")
print(df.head(2))
df.to_csv("output.csv")

In [ ]:
import pandas as pd
import numpy as np

# Load your processed file
df = pd.read_csv("processed_features_conf_0.5.csv", index_col="image_name")

print("=== CENTROID / PERIMETER QA CHECK ===\n")

# --- collect columns ---
cx_cols = [c for c in df.columns if c.startswith("seg_") and c.endswith("_cx")]
cy_cols = [c for c in df.columns if c.startswith("seg_") and c.endswith("_cy")]
perim_cols = [c for c in df.columns if c.startswith("seg_") and c.endswith("_perimeter")]

print(f"cx columns: {len(cx_cols)}")
print(f"cy columns: {len(cy_cols)}")
print(f"perimeter columns: {len(perim_cols)}\n")

# --- centroid checks ---
for name, cols in [("cx", cx_cols), ("cy", cy_cols)]:
    if not cols:
        print(f"[WARN] No {name} columns found.")
        continue

    vals = df[cols]

    out_of_range = (vals < 0) | (vals > 1)
    bad_count = out_of_range.sum().sum()
    nan_count = vals.isna().sum().sum()
    inf_count = np.isinf(vals.to_numpy()).sum()

    print(f"--- {name.upper()} ---")
    print(f"nan: {nan_count}")
    print(f"inf: {inf_count}")
    print(f"out of [0,1]: {bad_count}")

    if bad_count > 0:
        bad_locs = np.where(out_of_range)
        for i in range(min(5, len(bad_locs[0]))):
            row = vals.index[bad_locs[0][i]]
            col = vals.columns[bad_locs[1][i]]
            print(f"  {row} -> {col} = {df.loc[row, col]}")

    print()

# --- perimeter checks ---
if perim_cols:
    perim = df[perim_cols]

    negative = perim < 0
    nan_count = perim.isna().sum().sum()
    inf_count = np.isinf(perim.to_numpy()).sum()
    neg_count = negative.sum().sum()

    print("--- PERIMETER ---")
    print(f"nan: {nan_count}")
    print(f"inf: {inf_count}")
    print(f"negative: {neg_count}")

    # optional summary stats
    print("\nPerimeter stats:")
    print(perim.describe().T[["mean", "std", "min", "max"]].head(10))

    if neg_count > 0:
        bad_locs = np.where(negative)
        for i in range(min(5, len(bad_locs[0]))):
            row = perim.index[bad_locs[0][i]]
            col = perim.columns[bad_locs[1][i]]
            print(f"  {row} -> {col} = {df.loc[row, col]}")
else:
    print("[WARN] No perimeter columns found.")

print("\n=== DONE ===")

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("processed_features_conf_0.5.csv")

# Filter the image row
img_name = "40_90.jpg"
row = df[df["image_name"] == img_name]

if row.empty:
    print("Image not found")
else:
    row = row.iloc[0]  # get first matching row

    # Select relevant columns
    selected_cols = [
        col for col in df.columns
        if (col.endswith("cx") or col.endswith("cy") or "perim" in col)
    ]

    # Print non-NaN values only
    for col in selected_cols:
        value = row[col]
        if pd.notna(value):
            print(f"{col}: {value}")